# Eva LoRA on Google Colab (free GPU)

1. **Runtime → Change runtime type → Hardware accelerator → GPU → Save**
2. Upload your `eva-kb-alpaca-*.jsonl` (from `npm run eva:export-train`)
3. Run all cells
4. Download `eva-lora-peft.zip` → on PC run merge/ollama as usual

Free tier: GPU not guaranteed, sessions disconnect when idle, daily/weekly quotas apply.

In [ ]:
# Check GPU (should show Tesla T4 / similar on free)
!nvidia-smi
import torch
print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

In [ ]:
# Install (Colab usually has torch+cuda already; add peft stack)
%pip install -q -U transformers datasets accelerate peft trl sentencepiece protobuf

In [ ]:
from google.colab import files
print('Upload your Alpaca JSONL (eva-kb-alpaca-....jsonl)')
uploaded = files.upload()
DATA = next(iter(uploaded))
print('Using', DATA)

In [ ]:
import json
from pathlib import Path

BASE = "Qwen/Qwen2.5-3B-Instruct"  # free T4 can usually handle 3B QLoRA; use 0.5B if OOM
# BASE = "Qwen/Qwen2.5-0.5B-Instruct"
OUT = Path("eva-lora-peft")
MAX_STEPS = 60
MAX_SEQ = 512

rows = []
with open(DATA, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            rows.append(json.loads(line))
print(f"examples={len(rows)} base={BASE}")
assert len(rows) >= 5

In [ ]:
import torch
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

use_4bit = True  # QLoRA — fits free T4 better for 3B
bnb = None
if use_4bit:
    %pip install -q bitsandbytes
    from transformers import BitsAndBytesConfig
    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

tokenizer = AutoTokenizer.from_pretrained(BASE, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE,
    trust_remote_code=True,
    quantization_config=bnb,
    device_map="auto",
)
if use_4bit:
    model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

def to_text(ex):
    return {
        "text": f"System: {ex.get('system','')}\nUser: {ex.get('instruction','')}\nAssistant: {ex.get('output','')}"
    }

ds = Dataset.from_list(rows).map(to_text, remove_columns=list(rows[0].keys()))
OUT.mkdir(exist_ok=True)

args = SFTConfig(
    output_dir=str(OUT),
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    max_steps=MAX_STEPS,
    logging_steps=5,
    fp16=True,
    optim="paged_adamw_8bit" if use_4bit else "adamw_torch",
    report_to=[],
    dataset_text_field="text",
    max_length=MAX_SEQ,
    packing=False,
)

trainer = SFTTrainer(model=model, args=args, train_dataset=ds, processing_class=tokenizer)
trainer.train()
trainer.model.save_pretrained(str(OUT))
tokenizer.save_pretrained(str(OUT))
print("saved", OUT.resolve())

In [ ]:
!zip -r eva-lora-peft.zip eva-lora-peft
from google.colab import files
files.download("eva-lora-peft.zip")

## Back on your PC

```bash
# unzip into training/out/eva-lora-peft
npm run eva:merge-lora -- -SkipMerge   # if you only need convert+ollama after local merge
# or merge from adapter:
# .\.venv-lora\Scripts\python.exe training\merge_lora_peft.py --adapter training\out\eva-lora-peft --out training\out\eva-lora-merged
npm run eva:merge-lora
```

Set `OLLAMA_MODEL=eva-lora` and restart Eva.